# 52. Both BatchNorm routes on the saved hypernetwork models

Notebook 51 read an ordinary ResNet50 two ways at every boundary. Read through
the statistics the network built while training, effective rank rose from 7.30
to 115.66. Read through the statistics a freshly built ResNet50 has, a running
mean of 0 and a running variance of 1 that never change, the same network at
the same boundaries fell from 7.30 to 1.18.

The study only ever reads the second way. `Trainer.__init__` clones the buffers
of the target as `build_target` returns it, and `build_target` calls
`resnet50(weights=None)`, so `trainer.buffer_template` holds those untouched
initialisation values. Every published indicator number in this study comes
through that route.

So the hypernetwork side of the table has one column and the ordinary side has
two. This notebook fills in the missing column. It takes the saved order A
models, generates the ResNet50 weights for each task code, measures what those
weights actually do to the statistics of the learned groups' pictures, and
reads the indicators again through the measured statistics.

Nothing trains. This is forward passes only.

In [ ]:
from pathlib import Path
import sys, os, json, time, subprocess, importlib.util

IN_COLAB = "google.colab" in sys.modules

# Importing uncle twice in one session gives two copies of the module with two
# copies of its constants, which is a confusing way to fail much later.
if any(name == "uncle" or name.startswith("uncle.") for name in sys.modules):
    raise RuntimeError("Restart the runtime before loading the code.")

if IN_COLAB:
    REPO = Path("/content/path-indicators")
    if not REPO.exists():
        subprocess.run(["git", "clone",
                        "https://github.com/sumitasthana/PATH.git", str(REPO)],
                       check=True)
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "uncle/task_partition_10x20.json").exists()), None)
    if REPO is None:
        raise FileNotFoundError("Open this from a checkout that has the ten-group split.")

for package in ("boto3", "tqdm"):
    if importlib.util.find_spec(package) is None and IN_COLAB:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", package],
                       check=True)
sys.path.insert(0, str(REPO))

import torch
from torch import nn
from torch.func import functional_call
from tqdm.auto import tqdm
from uncle import checkpoint
from uncle.config import Config, PROBE_TASK, study_defaults
from uncle.hypernet import HyperNetwork, build_target
from uncle.plasticity_indicators import (
    SRANK_DELTA, THRESHOLDS, activation_sites, dormant_fraction,
    fixed_image_sets, reference_code)
from uncle.probe import _features_and_logits
from uncle.storage import R2Store
from uncle.streams import build_tasks
from uncle.tasks import PARTITIONS
from uncle.trainer import UnCLe

HEAD = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"],
                               text=True).strip()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu"
print("Running at:", HEAD)
print("Device:", GPU_NAME)

## Settings

Order A only, five seeds. That is the set notebook 51 compares against and the
set every indicator figure in the draft uses.

Two boundaries. Boundary 9 is the end of the stream and is what the question
asks for. Boundary 0 is added because it costs nothing: the untrained generator
is rebuilt from the seed and no file is downloaded. Without it there is no
starting value, and the table cannot have the change column that notebook 51's
table has.

In [ ]:
STUDY_ID = "plasticity_revised_v1"
BUCKET = "uncle-experiments"

READ_ORDERS = ["order_A"]
READ_SEEDS = [0, 1, 2, 3, 4]

# Boundary 0 needs no download. Boundary 9 is one 540 MB file per seed, and
# each file is deleted as soon as it has been read.
READ_BOUNDARIES = [0, 9]

# The pictures the indicators are read on. 256 per set, the same ones every
# published reading in this study used, drawn from the held-out split.
IMAGES_PER_SET = 256

# The pictures the BatchNorm statistics are measured on. These come from the
# nine learned groups' training split, so they do not overlap the 256 pictures
# above. 2048 is 32 batches of 64, which is enough for a stable mean and
# variance at every layer without making the run long.
ADAPT_IMAGES = 2048
BATCH_SIZE = 64

# The task each ordering unlearns at the end. Needed to rebuild a run's
# configuration, because the request list is part of what names the run.
FORGET = {"order_A": "0", "order_B": "4", "order_C": "8"}

SAVE_KEY = (f"uncle/indicator_protocol/{STUDY_ID}/"
            "buffer_route_on_hypernetwork_v5.jsonl")

# Notebook 51's results, read back so the final table can put the two networks
# side by side without any number being retyped.
# v2 carries the dormant unit count; v1 does not. The newer file is used
# when it exists, so this notebook still runs before notebook 51 is re-run and
# simply leaves the dormant row for the ordinary network empty.
PLAIN_KEYS = [f"uncle/indicator_protocol/{STUDY_ID}/plain_network_stream_v2.jsonl",
              f"uncle/indicator_protocol/{STUDY_ID}/plain_network_stream.jsonl"]

DATA_ROOT = (Path("/content/data/tiny-imagenet-200") if IN_COLAB
             else REPO / "data/tiny-imagenet-200")
CACHE = Path("/content/indicators") if IN_COLAB else REPO / "outputs/indicator-protocol"
CACHE.mkdir(parents=True, exist_ok=True)

READ_RUNS = [(o, s) for o in READ_ORDERS for s in READ_SEEDS]
downloads = len(READ_RUNS) * len([p for p in READ_BOUNDARIES if p > 0])
print(f"{len(READ_RUNS)} runs x {len(READ_BOUNDARIES)} boundaries, "
      f"{downloads} model files to download (about {downloads * 0.54:.1f} GB)")
print("No training. Forward passes only.")

## Pictures

Two jobs, two sets of pictures, and they must not overlap.

The reading sets are the same 256 pictures per set that every published reading
in this study used. They come from the held-out split, so no model trained on
them.

The adaptation set is separate. It is 2048 pictures from the nine learned
groups' training split. Its only job is to let each BatchNorm layer measure the
mean and variance that this model's own weights produce. Using the reading
pictures for that job would let the statistics be fitted to the very pictures
the indicator is then computed on.

In [ ]:
def secret(name):
    """Read a credential from Colab secrets, or the environment elsewhere."""
    if IN_COLAB:
        from google.colab import userdata
        return userdata.get(name)
    value = os.environ.get(name)
    if value is None:
        env = REPO / ".env"
        if env.exists():
            for line in env.read_text(encoding="utf-8").splitlines():
                if line.strip().startswith(name + "="):
                    return line.partition("=")[2].strip().strip(chr(34)).strip(chr(39))
    return value


store = R2Store.connect(secret("R2_ENDPOINT"), BUCKET,
                        secret("R2_ACCESS_KEY_ID"), secret("R2_SECRET_ACCESS_KEY"))

# The settings the streams were trained with, read back rather than retyped, so
# the model this notebook rebuilds has the shape its checkpoint expects.
plan = json.loads(store.download(
    f"uncle/indicators/{STUDY_ID}/job2_summary.json", CACHE).read_text(encoding="utf-8"))
CLASSES = plan["classes_per_task"]
print("Streams used learning rate", plan["learning_rate"], "| epochs", plan["epochs"],
      "| batch", plan["batch_size"], "| classes per task", CLASSES)

# Fail now rather than ten minutes in, if a model this session needs is missing.
missing = []
for order, seed in READ_RUNS:
    for point in READ_BOUNDARIES:
        if point == 0:
            continue
        key = f"uncle/checkpoints/{STUDY_ID}/{order}/seed_{seed}/boundary_{point}.pt"
        if store.info(key) is None:
            missing.append(key)
if missing:
    raise RuntimeError("Missing models:\n" + "\n".join(sorted(set(missing))))
print("Every model this session needs is in R2.")

## The two routes

`start` is the study's route. The buffers are `trainer.buffer_template`: a
running mean of 0 and a running variance of 1 at every BatchNorm layer, which
is what a freshly built ResNet50 has. They are the same at every boundary, for
every code, in every run. That is why the study chose them. It means a change
in the features can only come from the weights.

`own` is the other route. The generated weights are run over 2048 pictures from
the learned groups with the BatchNorm layers in training mode, so each layer
measures the mean and variance that these particular weights actually produce.
Then the indicators are read through those measured values.

A hypernetwork writes a fresh set of weights for every task code, so it has no
single set of running statistics that belongs to it. That is the honest reason
the fixed template was chosen. This notebook does not argue the choice was
wrong. It measures how much the choice changes the answer.

In [ ]:
from torch.utils.data import ConcatDataset

STREAM_TASKS = [str(i) for i in range(9)]
needed = tuple(STREAM_TASKS + [PROBE_TASK])
tasks = build_tasks(root=DATA_ROOT, partition_path=PARTITIONS[20],
                    partition_seed=42, include=needed, download=True)
print("Loaded Tiny ImageNet images for", len(needed), "tasks")

# The reading sets. in_stream is pictures from groups the models learned, and
# probe is the held-out group 9 that no model ever trained on.
sources = {
    "in_stream": ConcatDataset([tasks[name]["test"] for name in STREAM_TASKS]),
    "probe": tasks[PROBE_TASK]["test"],
}
image_sets = fixed_image_sets(sources, count=IMAGES_PER_SET)

# The adaptation set, from the training split of the same nine groups. The
# drawing is seeded inside fixed_image_sets, so every seed and every boundary
# measures its statistics on exactly the same pictures and only the model
# differs.
adapt_source = ConcatDataset([tasks[name]["train"] for name in STREAM_TASKS])
adapt_images = fixed_image_sets({"adapt": adapt_source},
                                count=ADAPT_IMAGES)["adapt"]

for name, images in image_sets.items():
    print(f"read   {name:<10} {tuple(images.shape)}")
print(f"adapt  {'train':<10} {tuple(adapt_images.shape)}")

## Run

Every finished model is written to R2 before the next one starts, and a model
already in the file is skipped. So a stopped run costs only the model it was in
the middle of.

In [ ]:
import random
import numpy as np


def build(order, seed, point):
    """The model this stream had at `point`. Point 0 needs no file.

    Point 0 is the untrained generator. It is rebuilt from the seed rather than
    loaded, which is why no boundary-0 file exists anywhere in this study.
    """
    settings = study_defaults(order)
    settings["requests"] = settings["requests"] + (("forget", FORGET[order]),)
    config = Config(**settings, seed=seed,
                    learning_rate=plan["learning_rate"], epochs=plan["epochs"],
                    batch_size=plan["batch_size"],
                    initialization=plan["initialization"], device=DEVICE)
    # The generator's starting weights come from the global random number
    # generator, not from config.seed by itself: HyperNetwork.__init__ calls
    # kaiming_normal_, normal_ and randn, and all three read the global state.
    # uncle/paired_study.py seeds these before it builds, so a rebuilt model
    # only matches the study's unless this does the same. At a loaded boundary
    # the checkpoint overwrites all of it, so this matters only for point 0.
    random.seed(config.seed)
    np.random.seed(config.seed)
    torch.manual_seed(config.seed)
    if DEVICE == "cuda":
        torch.cuda.manual_seed_all(config.seed)
    target = build_target(config)
    hypernet = HyperNetwork(target, config)
    trainer = UnCLe(hypernet, config, target, tasks, progress=False)
    path = None
    if point > 0:
        key = f"uncle/checkpoints/{STUDY_ID}/{order}/seed_{seed}/boundary_{point}.pt"
        path = store.download(key, CACHE)
        payload = torch.load(path, map_location=DEVICE, weights_only=False)
        checkpoint.restore(payload, hypernet=hypernet, uncle=trainer)
        del payload
    return target, hypernet, trainer, path

## The table

Three measures, two routes, two boundaries. The hypernetwork rows are split by
code. `reference` is the fixed untrained code every published reading in this
study used. `trained codes` averages the nine codes the stream actually
learned, one value per seed first so a seed counts once and not nine times.
They are kept apart because notebook 48 already showed the reading moves by
about 1.4 times between them.

In [ ]:
def rank_pair(matrix, centre):
    """Effective rank and srank of one feature matrix.

    `matrix` has one row per picture and 2048 columns. Setting `centre` to True
    subtracts the average row first.

    Effective rank is the exponential of the entropy of the normalised singular
    values, from Roy and Vetterli 2007. srank counts how many singular values
    are needed to reach 99 percent of their total, from Kumar et al. 2021.
    Both are copied from the library rather than imported so that both buffer
    routes go through exactly the same arithmetic.
    """
    work = matrix.double()
    if centre:
        work = work - work.mean(dim=0, keepdim=True)
    values = torch.linalg.svdvals(work)
    total = values.sum()
    if total <= 0:
        return float("nan"), 0
    weights = values / total
    weights = weights[weights > 0]
    effective = float(torch.exp(-(weights * weights.log()).sum()).item())
    shares = torch.cumsum(values, 0) / total
    return effective, int((shares < 1 - SRANK_DELTA).sum().item()) + 1


def mean_share(matrix):
    """How much of the matrix is the one row every picture shares.

    Take the average row, measure its length, and compare that to the length of
    the whole matrix. The square root of the number of rows is there because
    the whole matrix stacks that many rows. The result runs from 0 to 1. Near 1
    means every picture gives almost the same features, which by itself drives
    the uncentred effective rank towards 1 whatever else the model does.
    """
    work = matrix.double()
    norm = float(work.norm())
    if norm <= 0:
        return float("nan")
    return float(work.mean(dim=0).norm() * (work.shape[0] ** 0.5) / norm)


@torch.no_grad()
def measured_buffers(target, trainer, weights):
    """BatchNorm statistics that match these generated weights.

    The study reads every model through `trainer.buffer_template`, which holds
    what a freshly built ResNet50 has: a running mean of 0 and a running
    variance of 1 at every BatchNorm layer, values that never change however
    far the weights travel. This function makes the other reading. It runs the
    generated weights over the adaptation pictures with the BatchNorm layers in
    training mode, so each layer measures the mean and variance its own weights
    actually produce on data from the learned groups.

    Setting `momentum` to None makes each layer keep a plain running average
    over every batch it sees, instead of an exponentially decaying one. The
    result then does not depend on the order the batches arrive in, which an
    exponential average would. The momentum values are put back afterwards.
    """
    buffers = {name: value.detach().clone()
               for name, value in trainer.buffer_template.items()}
    # Start every counter at zero so the running average covers these batches
    # and nothing else.
    for name, value in buffers.items():
        if name.endswith("num_batches_tracked"):
            value.zero_()

    saved = {}
    for name, module in target.named_modules():
        if isinstance(module, nn.modules.batchnorm._BatchNorm):
            saved[name] = module.momentum
            module.momentum = None

    was_training = target.training
    target.train()
    try:
        for start in range(0, len(adapt_images), BATCH_SIZE):
            batch = adapt_images[start:start + BATCH_SIZE].to(trainer.device)
            # strict=True means every parameter and buffer must be supplied, so
            # a name that stops matching fails here rather than silently using
            # the module's own value.
            functional_call(target, (weights, buffers), (batch,), strict=True)
    finally:
        target.train(was_training)
        for name, module in target.named_modules():
            if name in saved:
                module.momentum = saved[name]

    # Every BatchNorm layer should have seen every batch. If this fails, the
    # statistics are not an average over the data they claim to describe.
    counted = {int(value.item()) for name, value in buffers.items()
               if name.endswith("num_batches_tracked")}
    expected = (len(adapt_images) + BATCH_SIZE - 1) // BATCH_SIZE
    assert counted == {expected}, (counted, expected)
    return buffers


@torch.no_grad()
def read_dormant(hypernet, target, trainer, images, weights, buffers):
    """The share of units that have gone quiet, for one set of weights and buffers.

    A unit is called dormant when its mean absolute activation is small beside
    the mean over all units at the same place in the network. Sokar et al. 2023
    define it that way and report three cut-offs, which is what THRESHOLDS
    holds. This is the third indicator in the paper's own opening table, and
    the only one that had never been read through both buffer routes.

    The counting itself is the library's, so these numbers go through exactly
    the same arithmetic as the published ones. The one difference is that the
    buffers are supplied by the caller instead of always being the fixed
    template, which is the whole point of this notebook.

    Unlike effective rank, this reads every ReLU in the network rather than
    only the 2048 numbers at the classifier. If the fixed template is what
    makes units look dormant, it should show here most of all: a unit counts as
    dormant when its activation is small, and reading through statistics that
    do not match the weights is exactly what shrinks activations.
    """
    modes = hypernet.training, target.training
    hypernet.eval()
    target.eval()
    try:
        def run(reset):
            for start in range(0, len(images), BATCH_SIZE):
                # The sums are keyed by the position of each ReLU call, so the
                # counter has to go back to zero at the start of every batch.
                reset()
                _features_and_logits(hypernet, target, trainer, None,
                                     images[start:start + BATCH_SIZE].to(trainer.device),
                                     weights=weights, buffers=buffers)
        return dormant_fraction(activation_sites(target, run))
    finally:
        hypernet.train(modes[0])
        target.train(modes[1])


@torch.no_grad()
def read_pre_relu(hypernet, target, trainer, images, weights, buffers):
    """The same 2048 numbers, read one step earlier, before the last ReLU.

    Why this exists. Every number the study reads sits after a ReLU, so none of
    them can be negative. A ReLU replaces every negative value with zero. If
    reading a model through BatchNorm statistics that do not match its weights
    pushes the whole signal upward, then after the ReLU almost nothing is cut,
    every picture produces nearly the same row, and effective rank falls to
    near 1 for that reason alone. Reading before the ReLU, where negative
    values still exist, separates that explanation from the alternative, which
    is that the rescaling itself destroys the directions.

    How the place is found. A hook is put on every ReLU in the network and each
    one overwrites the same slot, so after the forward pass the slot holds the
    input of the last ReLU that ran. That is the one feeding the classifier.
    Hooks fire in the order the layers actually run, so this stays correct even
    though a ResNet bottleneck calls the same ReLU module three times.

    The hook has to be a PRE-hook, which runs before the layer, and it has to
    copy the tensor. A ResNet builds its ReLU layers with inplace=True, meaning
    the layer writes its result over its own input instead of making a new
    tensor. An ordinary hook runs after the layer, so by then that memory has
    already been clipped and reading it gives the output a second time, not the
    input. An earlier version of this function made exactly that mistake and
    every pre-ReLU number came back equal to the ordinary one.

    Two things come back. The first is the signal averaged over the picture's
    width and height, which is what the network's own pooling layer does, so
    the matrix has the same shape as the ordinary reading and the two are
    comparable. The second is the share of values that are negative, counted on
    the full map before any averaging, because that is the share the ReLU
    actually sets to zero. Counting it after averaging would hide it: a
    position that is negative in one corner and positive in another averages to
    a positive number while the ReLU still cut the corner.
    """
    last = {}

    def hook(_module, inputs):
        # The copy is not optional. The layer is about to overwrite this exact
        # memory, so a reference would end up holding the clipped values.
        last["x"] = inputs[0].detach().clone()

    handles = [module.register_forward_pre_hook(hook)
               for module in target.modules() if isinstance(module, nn.ReLU)]
    if not handles:
        raise RuntimeError("No ReLU found, so there is nothing to read before.")

    modes = hypernet.training, target.training
    hypernet.eval()
    target.eval()
    collected = []
    cut = seen = 0
    try:
        for start in range(0, len(images), BATCH_SIZE):
            batch = images[start:start + BATCH_SIZE].to(trainer.device)
            _features_and_logits(hypernet, target, trainer, None, batch,
                                 weights=weights, buffers=buffers)
            full = last["x"]
            cut += int((full < 0).sum())
            seen += full.numel()
            # Average over height and width, the same thing the network's own
            # pooling layer does, so this matrix lines up with the usual one.
            collected.append(full.mean(dim=(2, 3)).float().cpu())
    finally:
        for handle in handles:
            handle.remove()
        hypernet.train(modes[0])
        target.train(modes[1])
    return torch.cat(collected), cut / seen


@torch.no_grad()
def read_features(hypernet, target, trainer, images, weights, buffers):
    """The classifier's 2048 inputs, for one set of weights and one set of buffers.

    Evaluation mode is forced. A BatchNorm layer left in training mode rescales
    using whatever batch is in front of it, which revives a nearly dead signal
    and hides what is being measured. The previous modes are put back after.
    """
    modes = hypernet.training, target.training
    hypernet.eval()
    target.eval()
    collected = []
    try:
        for start in range(0, len(images), BATCH_SIZE):
            batch = images[start:start + BATCH_SIZE].to(trainer.device)
            # task is unused once weights and buffers are both supplied.
            captured, _ = _features_and_logits(hypernet, target, trainer, None,
                                               batch, weights=weights,
                                               buffers=buffers)
            collected.append(captured.float().cpu())
    finally:
        hypernet.train(modes[0])
        target.train(modes[1])
    return torch.cat(collected)

## What the table decides

**If the hypernetwork rises under `own` statistics the way the ordinary network
does**, then the fall in every published figure of this study is the buffer
route and nothing else. The rank section of the paper becomes a measurement
result, and the fixed template has to be reported as a choice with a size
attached to it.

**If the hypernetwork stays low under both routes**, then the two networks
genuinely differ, and the fixed template is not the whole story. That is a
different paper, and a more interesting one for the hypernetwork side, but it
needs this table to be claimed at all.

**Either way the mean row share decides the second question.** The ordinary
network holds 0.954 under `start` and drops to 0.740 under `own`. If the
hypernetwork holds near 0.98 under both, the shared-row signature belongs to
the generator. If it drops the same way, the signature belongs to the route.

One limit to state in the paper either way. A hypernetwork has no running
statistics of its own, so `own` here is a quantity this notebook constructed,
not one the model carries. It answers "what would these weights do if their
statistics matched them", which is the right question for comparing against an
ordinary network. It is not a claim that the study should have used it.

In [ ]:
def read_one_model(order, seed, point):
    """Every code, every buffer route, every picture set, for one saved model.

    The codes are the fixed reference code the published records use, plus
    every task code this stream actually trained. Each code generates its own
    ResNet50, so each code gets its own measured statistics as well. Reusing
    one code's statistics for another code's weights would mix the two things
    this notebook is trying to separate.
    """
    target, hypernet, trainer, path = build(order, seed, point)
    try:
        # task_codes is the dict the `known_tasks` property wraps. Reading it
        # directly keeps this independent of whether that stays a property.
        codes = {"reference": reference_code(hypernet)}
        for task in hypernet.task_codes:
            codes[f"task_{task}"] = hypernet.task_codes[task].detach()

        rows = []
        for code_name, code in codes.items():
            weights = hypernet.weights_from_code(code)
            routes = {"start": trainer.buffer_template,
                      "own": measured_buffers(target, trainer, weights)}
            for mode, buffers in routes.items():
                for set_name, images in image_sets.items():
                    matrix = read_features(hypernet, target, trainer, images,
                                           weights, buffers)
                    assert matrix.shape == (IMAGES_PER_SET, 2048), matrix.shape
                    raw_e, raw_s = rank_pair(matrix, centre=False)
                    mid_e, mid_s = rank_pair(matrix, centre=True)
                    quiet = read_dormant(hypernet, target, trainer, images,
                                         weights, buffers)
                    before, cut_share = read_pre_relu(hypernet, target, trainer,
                                                      images, weights, buffers)
                    assert before.shape == matrix.shape, (before.shape, matrix.shape)
                    pre_e, pre_s = rank_pair(before, centre=False)
                    rows.append(dict(
                        study=STUDY_ID, order=order, seed=seed, point=point,
                        code=code_name, trained_code=code_name != "reference",
                        buffers=mode, images=set_name,
                        effective_rank=raw_e, srank=raw_s,
                        effective_rank_centred=mid_e, srank_centred=mid_s,
                        mean_share=mean_share(matrix), **quiet,
                        effective_rank_pre_relu=pre_e, srank_pre_relu=pre_s,
                        mean_share_pre_relu=mean_share(before),
                        negative_share_pre_relu=cut_share,
                        n_images=int(matrix.shape[0]),
                        n_adapt_images=int(len(adapt_images)),
                        code_revision=HEAD, device=GPU_NAME))
        return rows
    finally:
        del trainer, hypernet, target
        import gc
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
        # One model file at a time, so five runs do not fill the disk.
        if path is not None and Path(path).exists():
            Path(path).unlink()

In [ ]:
reads = []
if store.info(SAVE_KEY) is not None:
    text = store.download(SAVE_KEY, CACHE).read_text(encoding="utf-8")
    reads = [json.loads(l) for l in text.splitlines() if l.strip()]
    print(f"Resuming: {len(reads)} readings already saved.")

PARTIAL = CACHE / "buffer_route_partial.jsonl"
PARTIAL.write_text("".join(json.dumps(r) + "\n" for r in reads), encoding="utf-8")

done = {(r["order"], r["seed"], r["point"]) for r in reads}
began = time.perf_counter()
for order, seed in tqdm(READ_RUNS, desc="runs", unit="run"):
    for point in READ_BOUNDARIES:
        if (order, seed, point) in done:
            continue
        rows = read_one_model(order, seed, point)
        reads += rows
        with PARTIAL.open("a", encoding="utf-8") as handle:
            for row in rows:
                handle.write(json.dumps(row) + "\n")
        store.upload(PARTIAL, SAVE_KEY, overwrite=True)
        print(f"  {order} seed {seed} boundary {point}: {len(rows)} readings")
print(f"\n{len(reads)} readings in {(time.perf_counter() - began) / 60:.1f} minutes.")

In [ ]:
from statistics import mean, stdev


def one_value_per_seed(rows, field, **want):
    """One number per seed, averaging that seed's codes first.

    A seed contributes nine trained codes. Averaging all the rows together
    would let a seed count nine times and make the spread across seeds look
    far smaller than it is.
    """
    chosen = [r for r in rows
              if all(r.get(key) == value for key, value in want.items())]
    by_seed = {}
    for row in chosen:
        # A file written before a measure existed simply has no such key.
        # Notebook 51 predates the dormant unit count, so its rows are skipped
        # for that measure and shown as a dot rather than stopping the table.
        if field in row:
            by_seed.setdefault(row["seed"], []).append(row[field])
    return [mean(values) for values in by_seed.values()]


def cell(values):
    """Mean across seeds, or a dot when there is nothing to show."""
    return f"{mean(values):9.3f}" if values else "        ."


MEASURES = [("effective_rank", "effective rank"),
            ("srank", "srank"),
            ("mean_share", "mean row share"),
            ("effective_rank_centred", "centred effective rank"),
            (f"dormant_{THRESHOLDS[0]}", f"dormant units, cut-off {THRESHOLDS[0]}"),
            ("effective_rank_pre_relu", "effective rank, before the last ReLU"),
            ("mean_share_pre_relu", "mean row share, before the last ReLU"),
            ("negative_share_pre_relu", "share of values the ReLU would cut")]
IMAGES = "in_stream"
CODE_GROUPS = [("reference code", dict(code="reference")),
               ("trained codes", dict(trained_code=True))]

for field, label in MEASURES:
    print()
    print(f"{label.upper()} on {IMAGES} pictures")
    print(f"{'what':<36} {'own stats':>11} {'start stats':>13}  seeds")
    for point in READ_BOUNDARIES:
        for code_label, want in CODE_GROUPS:
            columns = {}
            for mode in ("own", "start"):
                columns[mode] = one_value_per_seed(
                    reads, field, point=point, buffers=mode, images=IMAGES,
                    **want)
            if not columns["own"] and not columns["start"]:
                continue
            name = f"hypernetwork t={point}, {code_label}"
            print(f"{name:<36} {cell(columns['own']):>11} "
                  f"{cell(columns['start']):>13}  {len(columns['own'])}")

In [ ]:
# Notebook 51's ordinary network, read back from its own file so the two
# networks sit in one table without a number being copied by hand.
plain = []
for candidate in PLAIN_KEYS:
    if store.info(candidate) is None:
        continue
    text = store.download(candidate, CACHE).read_text(encoding="utf-8")
    plain = [json.loads(l) for l in text.splitlines() if l.strip()]
    has_dormant = any(f"dormant_{THRESHOLDS[0]}" in row for row in plain)
    print(f"Notebook 51: {len(plain)} readings from {candidate.split('/')[-1]}"
          f"{'' if has_dormant else ', no dormant count in this file'}")
    break
if not plain:
    print("Notebook 51's file was not found, so only the hypernetwork is shown.")

LAST = len(STREAM_TASKS)

# Each row of the joint table: a name, where its rows come from, which rows to
# keep, and what that file calls the boundary. Notebook 51 writes "t" and this
# notebook writes "point", which is the only difference between the two files.
JOINT_ROWS = [("hypernetwork, reference code", "reads", dict(code="reference"), "point"),
              ("hypernetwork, trained codes", "reads", dict(trained_code=True), "point")]
if plain:
    JOINT_ROWS.append(("ordinary ResNet50", "plain", {}, "t"))

for field, label in MEASURES:
    print()
    print(f"{label.upper()} on {IMAGES} pictures, both networks")
    print(f"{'network':<30} {'t=0 own':>9} {'t=9 own':>9} "
          f"{'t=0 start':>10} {'t=9 start':>10} {'own change':>11}")
    for name, source_name, want, time_field in JOINT_ROWS:
        source = reads if source_name == "reads" else plain
        values = {}
        for mode in ("own", "start"):
            for moment in (0, LAST):
                values[(mode, moment)] = one_value_per_seed(
                    source, field, buffers=mode, images=IMAGES,
                    **{**want, time_field: moment})
        change = "          ."
        if values[("own", 0)] and values[("own", LAST)]:
            change = f"{mean(values[('own', LAST)]) - mean(values[('own', 0)]):+11.3f}"
        print(f"{name:<30} {cell(values[('own', 0)]):>9} "
              f"{cell(values[('own', LAST)]):>9} "
              f"{cell(values[('start', 0)]):>10} "
              f"{cell(values[('start', LAST)]):>10} {change}")

print()
print("own stats   = BatchNorm statistics measured from the model's own weights")
print("              on 2048 pictures from the nine learned groups")
print("start stats = the study's route: running mean 0, running variance 1,")
print("              the values a freshly built ResNet50 has, never updated")